# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. 
2. 
3. 
4. 
5. 

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [3]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [ ]:
# 1
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5463 entries, 0 to 5462
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       5463 non-null   int64  
 1   order_date     5463 non-null   str    
 2   customer_id    5463 non-null   str    
 3   city           5463 non-null   str    
 4   category       5463 non-null   str    
 5   amount         5463 non-null   float64
 6   delivery_days  5303 non-null   float64
 7   status         5463 non-null   str    
dtypes: float64(2), int64(1), str(5)
memory usage: 341.6 KB


In [6]:
df.shape[0] - df["delivery_days"].count()

np.int64(160)

In [19]:
print(df["city"].unique())
print(df["category"].unique())
print(df["status"].unique())
print(df["customer_id"].unique())

<StringArray>
[   'Алматы',    'Астана', 'Караганда',   'Шымкент',    'алматы',    'Атырау',
   'шымкент',  'Shymkent',    'астана',    'Актобе',   'Astana ',    'АЛМАТЫ',
   ' Алматы',    'Almaty']
Length: 14, dtype: str
<StringArray>
['Одежда', 'Красота', 'Дом', 'Книги', 'Электроника']
Length: 5, dtype: str
<StringArray>
['оплачен', 'отменён']
Length: 2, dtype: str
<StringArray>
['C00681', 'C01210', 'C01408', 'C01529', 'C00657', 'C00763', 'C01730',
 'C00896', 'C01349', 'C01934',
 ...
 'C00042', 'C00468', 'C01308', 'C01960', 'C00430', 'C00645', 'C01442',
 'C00018', 'C01650', 'C01057']
Length: 2003, dtype: str


<StringArray>
['C00681', 'C01210', 'C01408', 'C01529', 'C00657', 'C00763', 'C01730',
 'C00896', 'C01349', 'C01934',
 ...
 'C00042', 'C00468', 'C01308', 'C01960', 'C00430', 'C00645', 'C01442',
 'C00018', 'C01650', 'C01057']
Length: 2003, dtype: str

## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [25]:
# 2
df_notest = df[(df["customer_id"] != "TEST") & (df["status"] == "оплачен")]

df_notest["amount"].describe()

count      4901.000000
mean      14672.391349
std       13249.340787
min         720.000000
25%        5880.000000
50%       10300.000000
75%       18690.000000
max      139040.000000
Name: amount, dtype: float64

## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [49]:
# 3
sh = ["Шымкент", "шымкент", "Shymkent"]
ala = ["Алматы", "алматы", "Almaty"]
df_sh = df["city"].isin(sh)
df_ala = df["city"].isin(ala)

cancel_all = (df["status"] == "отменён").sum() / len(df)

cancel_sh = (((df["status"] == "отменён") & df_sh).sum() / df_sh.sum())
cancel_ala = (((df["status"] == "отменён") & df_ala).sum() / df_ala.sum())


print(cancel_all )
print(cancel_sh )
print(cancel_ala)

0.1023247300018305
0.20398009950248755
0.08106796116504854


## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [51]:
# 4

df_notest["month"] = df_notest["order_date"].str[:7]
df_task = df_notest[(df_notest["city"].isin(sh + ala)) & (df_notest["month"].isin(["2026-09", "2026-10"]))]

# df_task = df_notest[df_notest["month"].isin(["2026-09", "2026-10"])]
df_task.groupby(['city','month'])['amount'].sum()



city      month  
Almaty    2026-09      361430.0
          2026-10      746600.0
Shymkent  2026-09      277530.0
          2026-10      289350.0
Алматы    2026-09    13638620.0
          2026-10    12975290.0
Шымкент   2026-09     4264260.0
          2026-10     3517540.0
алматы    2026-09      510190.0
          2026-10      267160.0
шымкент   2026-09      408010.0
          2026-10      237120.0
Name: amount, dtype: float64

Использовал groupBy, что бы суммировать amount по городам и по месяцу. как в sql 

## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [54]:
# 5
cancel_sh = df[(df["city"].isin(sh)) & (df["status"] == "отменён")]
print(len(cancel_sh))

164


1) несколько условии в круглых скобках
2) Не везде город Шымкент записан так правильно. В некоторых полях он либо с маленькой буквы, либо на английском

## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [61]:
# 6
df_rates = df["customer_id"].value_counts()
df_rates

customer_id
C01818    11
C01076     9
C01947     9
C01656     9
C00420     9
          ..
C00645     1
C01442     1
C00018     1
C01650     1
C01057     1
Name: count, Length: 2003, dtype: int64

In [68]:
best_customer = df_rates.index[0]
mask = df["customer_id"] == best_customer
df.loc[mask]

,order_id,order_date,customer_id,city,category,amount,delivery_days,status,month
57,101125,2026-09-01 15:00,C01818,Шымкент,Дом,7350.0,1.0,оплачен,2026-09
779,100096,2026-09-09 18:00,C01818,Шымкент,Электроника,45270.0,3.0,оплачен,2026-09
1371,101504,2026-09-15 21:00,C01818,Алматы,Одежда,8890.0,1.0,оплачен,2026-09
1375,101504,2026-09-15 21:00,C01818,Алматы,Одежда,8890.0,1.0,оплачен,2026-09
2195,101860,2026-09-24 18:00,C01818,Астана,Красота,1490.0,1.0,оплачен,2026-09
2617,101462,2026-09-29 09:00,C01818,Астана,Электроника,22170.0,2.0,оплачен,2026-09
2776,101495,2026-09-30 21:00,C01818,Алматы,Электроника,20180.0,2.0,оплачен,2026-09
3417,103416,2026-10-08 14:00,C01818,Караганда,Дом,4330.0,1.0,оплачен,2026-10
4279,102644,2026-10-18 08:00,C01818,Астана,Дом,13600.0,3.0,оплачен,2026-10
4610,104001,2026-10-21 19:00,C01818,Алматы,Одежда,21990.0,1.0,оплачен,2026-10


## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [ ]:
# 7

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [ ]:
# 8